In [ ]:
import numpy as np
import xarray as xr
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
def angle_to_l(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 180. / x[~near_zero]
    return x


In [ ]:
def freq_to_time(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 1. / x[~near_zero]
    return x


In [ ]:
# grid
ntrunc=72
nSampWin = 720
spd = 2
nWindow = 59
frequency = np.fft.fftfreq(nSampWin, 1./spd)
ff = np.fft.fftshift(frequency[:])
ll = np.arange(0, ntrunc+1, 1)

In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
file_name = "olr-2xdaily-1981-2010-space-time-analysis-window-360-skip-180.nc"
ds_observed = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_observed = ds_observed.Fwflm_real.values[:, :, :, :] + 1j * ds_observed.Fwflm_imag.values[:, :, :, :]
Fwtlm_observed = ds_observed.Fwtlm_real.values[:, :, :, :] + 1j * ds_observed.Fwtlm_imag.values[:, :, :, :]


In [ ]:
Fwflm_observed = np.fft.fftshift(Fwflm_observed, axes=1)

In [ ]:
# angular variance -- observed
Cl_observed = np.zeros(ntrunc+1)
Clm_observed = np.zeros((ntrunc+1, 2*ntrunc+1))

for l in range(ntrunc+1):
    Cl_observed[l] = np.mean(np.mean(np.mean(np.abs(Fwtlm_observed[:, :, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=0), axis=0)
    Clm_observed[l, ntrunc-l:ntrunc+l+1] = np.mean(np.mean(np.abs(Fwtlm_observed[:, :, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=0)

# compensating for hanning window
Cl_observed *= 8. / 3.
Clm_observed *= 8. / 3.


In [ ]:
# because the variance changes by 2 orders of magnitudes it's better to fit the scaled data

In [ ]:
# cmb scaling
for l in range(ntrunc+1):
    Cl_observed[l] = l*(l+1) / (2*np.pi) * Cl_observed[l]
    Clm_observed[l] = l*(l+1) / (2*np.pi) * Clm_observed[l]

In [ ]:
# fit angular variance
# a**2 / (1 + b**2 * l(l+1))**2 * l*(l+1) / 2pi

popt_l, pcov_l = curve_fit(lambda l, a, b: a**2 * l * (l+1) / (2*np.pi) / ( 1 + b**2 * l * (l+1) )**2, ll[20:], Cl_observed[20:], p0=[5.5, 0.05])
perr_l = np.sqrt(np.diag(pcov_l))

Cl_ebcm = popt_l[0]**2 * ll * (ll+1) / (2*np.pi) / ( 1 + popt_l[1]**2 * ll * (ll+1) )**2

print(popt_l[0], perr_l[0])
print(popt_l[1], perr_l[1])

In [ ]:
fig_size = (14.00/2.54, 08.50/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 01.25, 12.00, 06.00])))


##### ax0 #####

for m in range(-ntrunc, ntrunc+1):
    ll = np.arange(np.abs(m), ntrunc+1, 1)

    if m == 0:
        ax[0].plot(ll, Clm_observed[np.abs(m):ntrunc+1, ntrunc+m], linestyle='', marker='.', color='C9', label=r'Samples with $|m| \leq \ell$', alpha=1)
    else:
        ax[0].plot(ll, Clm_observed[np.abs(m):ntrunc+1, ntrunc+m], linestyle='', marker='.', color='C9', alpha=1)


ll = np.arange(0, ntrunc+1, 1)

ax[0].plot(ll, Cl_observed, linestyle='', marker='.', color='C0', label=r'Average over $|m| \leq \ell$', alpha=1)


for m in [14]:
    ll = 15 #np.abs(m)
    ax[0].plot(ll, Clm_observed[ll, ntrunc+m], linestyle='', marker='.', color='C1', label=r'$(\ell, m) = (15, \pm 14)$', alpha=1)

for m in [15]:
    ll = 15 #np.abs(m)
    ax[0].plot(ll, Clm_observed[ll, ntrunc+m], linestyle='', marker='.', color='C2', label=r'$(\ell, m) = (15, \pm 15)$', alpha=1)


for m in [13]:
    ll = 13 #np.abs(m)
    ax[0].plot(ll, Clm_observed[ll, ntrunc+m], linestyle='', marker='.', color='C3', label=r'$(\ell, m) = (13, \pm 13)$', alpha=1)



secax = ax[0].secondary_xaxis('top', functions=(angle_to_l, angle_to_l))
secax.set_xlabel('Angular scale')
# secax.set_xticks(np.array([90, 20, 10, 7, 6, 5, 4, 3]))
secax.set_xticks(np.array([18, 9, 6, 4.5, 3.6, 3]))
secax.xaxis.set_major_formatter(StrMethodFormatter(u"{x:.1f}°"))
secax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')

# ax[0].set_title('Angular power spectrum', fontweight='bold')
ax[0].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[0].set_ylabel(r'$\ell (\ell+1) \, C_{\ell} \, / \, 2\pi$  $\mathrm{ [W \, m^{-2}]^{2}}$', fontsize=10)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
# ax[0].set_xscale('log')
# ax[0].set_yscale('log')
# ax[0].set_xlim(1, 80)
ax[0].set_ylim(None, 900)

# ax[0].grid()
ax[0].legend(ncol=1, frameon=True, edgecolor='grey')


In [ ]:
file_name = "fig-s04"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')